## Omnichannel Retail Optimization & Location Intelligence Platform

In [ ]:
import pandas as pd
import numpy as np

# ตั้งค่า Seed เพื่อให้สุ่มกี่ครั้งก็ได้ผลลัพธ์เท่าเดิม
np.random.seed(42)

# =====================================================================
# 1. GENERATE MOCK DATA (จำลองข้อมูลดิบ)
# =====================================================================

n_rows = 1000
channels = ['Offline Store', 'Online Website', 'Roboshop']
regions = ['Bangkok', 'Central', 'North', 'South', 'East']

# จำลองตารางยอดขาย (Sales Orders)
sales_data = pd.DataFrame({
    'order_id': [f"ORD_{1000 + i}" for i in range(n_rows)],
    'date': pd.date_range(start='2026-01-01', periods=n_rows, freq='h'),
    'channel': np.random.choice(channels, size=n_rows, p=[0.5, 0.3, 0.2]),
    'region': np.random.choice(regions, size=n_rows),
    'revenue': np.random.uniform(50, 1500, size=n_rows).round(2),
    'quantity': np.random.randint(1, 6, size=n_rows)
})

# ใส่ Missing Values ตั้งใจทำให้ข้อมูลสกปรกนิดนึงเพื่อจำลองการ Clean
sales_data.loc[np.random.choice(n_rows, size=20, replace=False), 'revenue'] = np.nan

# จำลองตารางค่าใช้จ่ายรายสาขา/ภูมิภาค (Expenses)
# ในชีวิตจริงข้อมูลต้นทุนมักจะมาเป็นรายเดือนหรือรายพื้นที่ ไม่ได้มาพร้อมยอดขาย
expense_data = pd.DataFrame({
    'region': regions,
    'leasing_cost': [150000, 50000, 40000, 45000, 42000],
    'marketing_cost': [80000, 30000, 25000, 28000, 26000],
    'operation_cost': [100000, 40000, 35000, 38000, 36000]
})

print("Raw Data Generated Successfully!")

# =====================================================================
# 2. DATA CLEANING & PROCESSING (ขั้นตอนจัดการข้อมูล)
# =====================================================================
print("\n--- Processing Data (ETL) ---")

# A. จัดการ Missing Values ในฝั่ง Revenue (ใช้ค่า Median ของช่องทางนั้น ๆ แทนเพื่อความสมเหตุสมผล)
sales_data['revenue'] = sales_data.groupby('channel')['revenue'].transform(lambda x: x.fillna(x.median()))

# B. คำนวณหา AOV (Average Order Value) ขั้นต้นในตารางยอดขาย
# (AOV = Revenue / Quantity ในบริบทของตะกร้าสินค้า)
sales_data['aov'] = (sales_data['revenue'] / sales_data['quantity']).round(2)

# C. จัดกลุ่ม (Aggregate) ยอดขายรายภูมิภาค เพื่อนำไปเช็กผลกำไรคู่กับค่าใช้จ่าย
regional_sales = sales_data.groupby('region').agg(
    total_revenue=('revenue', 'sum'),
    total_orders=('order_id', 'count'),
    total_qty=('quantity', 'sum')
).reset_index()

# D. MERGE DATA: รวมข้อมูลยอดขายและค่าใช้จ่ายเข้าด้วยกันด้วย SQL Concept (Inner Join)
final_report = pd.merge(regional_sales, expense_data, on='region', how='inner')

# E. คำนวณ Metrics สำคัญเพื่อตอบโจทย์ผู้บริหาร
final_report['total_expense'] = final_report['leasing_cost'] + final_report['marketing_cost'] + final_report['operation_cost']
final_report['net_profit'] = final_report['total_revenue'] - final_report['total_expense']
final_report['profit_margin_pct'] = ((final_report['net_profit'] / final_report['total_revenue']) * 100).round(2)

# =====================================================================
# 3. DISPLAY FINAL MASTER DATA
# =====================================================================
print("\n--- Final Master Table for Dashboard/Analysis ---")
print(final_report)

# เซฟไฟล์เป็น CSV เพื่อเอาไปใช้ทำสถิติต่อใน Phase ถัดไป และส่งต่อเข้า Power BI/Tableau
final_report.to_csv('regional_performance_master.csv', index=False)
sales_data.to_csv('cleaned_sales_data.csv', index=False)
print("\nFiles saved: 'regional_performance_master.csv' and 'cleaned_sales_data.csv'")

--- Generating Raw Data ---
Raw Data Generated Successfully!

--- Processing Data (ETL) ---

--- Final Master Table for Dashboard/Analysis ---
    region  total_revenue  total_orders  total_qty  leasing_cost  \
0  Bangkok     152710.090           193        566        150000   
1  Central     166591.975           212        621         50000   
2     East     167411.100           214        667         42000   
3    North     132989.490           173        539         40000   
4    South     160342.420           208        607         45000   

   marketing_cost  operation_cost  total_expense  net_profit  \
0           80000          100000         330000 -177289.910   
1           30000           40000         120000   46591.975   
2           26000           36000         104000   63411.100   
3           25000           35000         100000   32989.490   
4           28000           38000         111000   49342.420   

   profit_margin_pct  
0            -116.10  
1              27

## Regression and Clustering 

In [2]:
# import library 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# โหลดข้อมูลยอดขายและข้อมูลภูมิภาคที่เราคลีนไว้จาก Phase 1
# (เพื่อความง่ายในโปรเจกต์นี้ เราจะจำลองตารางคุณสมบัติของทำเลเพิ่มขึ้นมาเพื่อทำโมเดล)
np.random.seed(42)

# =====================================================================
# 1. CLUSTER ANALYSIS (จัดกลุ่มภูมิภาค/สาขา เพื่อจัดการต้นทุน)
# =====================================================================
print("\n[1/2] Running Cluster Analysis (K-Means)...")

# ดึงข้อมูล Master จาก Phase 1 มาใช้
df_master = pd.read_csv('regional_performance_master.csv')

# เลือก Feature ที่จะใช้จัดกลุ่ม: ในที่นี้คือ ยอดขายรวม (total_revenue) และ ค่าใช้จ่ายรวม (total_expense)
features = ['total_revenue', 'total_expense']
X_cluster = df_master[features]

# เนื่องจากสเกลของเงินต่างกัน ต้องทำ Feature Scaling ก่อนเข้า K-Means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# ทำ K-Means Clustering แยกออกเป็น 3 กลุ่ม (เช่น กลุ่ม Performance สูง, ปานกลาง, ต่ำ)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_master['cluster_id'] = kmeans.fit_predict(X_scaled)

print("Cluster Assignment Results:")
print(df_master[['region', 'total_revenue', 'total_expense', 'cluster_id']])


# =====================================================================
# 2. REGRESSION ANALYSIS (วิเคราะห์ปัจจัยเพื่อเลือก Store Location)
# =====================================================================
print("\n[2/2] Running Regression Analysis for Location Selection...")

# จำลองข้อมูลทำเลดิบ 50 โลเคชัน (ตึก/ห้างต่าง ๆ) เพื่อหาว่าปัจจัยไหนส่งผลต่อยอดขาย
n_locations = 50
location_data = pd.DataFrame({
    'location_id': [f"LOC_{100 + i}" for i in range(n_locations)],
    'foot_traffic_daily': np.random.randint(500, 5000, size=n_locations), # จำนวนคนเดินผ่านต่อวัน
    'competitor_count': np.random.randint(0, 8, size=n_locations),       # จำนวนร้านคู่แข่งในระยะ 1 กม.
    'marketing_spend': np.random.uniform(5000, 30000, size=n_locations),  # งบการตลาดที่อัดฉีด
    'monthly_revenue': np.zeros(n_locations)                              # ยอดขาย (เดี๋ยวจะคำนวณจากสมการบวกสุ่ม)
})

# สร้างความสัมพันธ์สมมุติ: ยอดขายจะโตตาม Traffic และงบการตลาด แต่จะลดลงถ้าคู่แข่งเยอะ
location_data['monthly_revenue'] = (
    (location_data['foot_traffic_daily'] * 120) + 
    (location_data['marketing_spend'] * 2.5) - 
    (location_data['competitor_count'] * 15000) + 
    np.random.normal(0, 30000, size=n_locations) # ใส่ noise ความไม่แน่นอนในโลกจริง
).round(2)

# กำหนดตัวแปรต้น (X) และตัวแปรตาม (Y)
X_reg = location_data[['foot_traffic_daily', 'competitor_count', 'marketing_spend']]
y_reg = location_data['monthly_revenue']

# เทรนโมเดล Linear Regression
reg_model = LinearRegression()
reg_model.fit(X_reg, y_reg)

# ดึงค่าสถิติสำคัญออกมาดู
r_sq = reg_model.score(X_reg, y_reg)
coefficients = reg_model.coef_
intercept = reg_model.intercept_

print(f"Model R-squared (ความแม่นยำของคำอธิบาย): {r_sq:.4f}")
print("Coefficients (อิทธิพลของแต่ละปัจจัยต่อยอดขาย):")
for col, coef in zip(X_reg.columns, coefficients):
    print(f" - {col}: {coef:.2f}")

# เซฟผลลัพธ์เป็นไฟล์ไว้ใช้ทำสไลด์และ Dashboard
location_data.to_csv('location_regression_analysis.csv', index=False)
print("\nAnalysis completed! File saved as 'location_regression_analysis.csv'")


[1/2] Running Cluster Analysis (K-Means)...
Cluster Assignment Results:
    region  total_revenue  total_expense  cluster_id
0  Bangkok     152710.090         330000           2
1  Central     166591.975         120000           0
2     East     167411.100         104000           0
3    North     132989.490         100000           1
4    South     160342.420         111000           0

[2/2] Running Regression Analysis for Location Selection...
Model R-squared (ความแม่นยำของคำอธิบาย): 0.9675
Coefficients (อิทธิพลของแต่ละปัจจัยต่อยอดขาย):
 - foot_traffic_daily: 116.50
 - competitor_count: -14601.97
 - marketing_spend: 3.59

Analysis completed! File saved as 'location_regression_analysis.csv'
